# Data Ingestion
## Source 1: OpenTargets
To query openTargets for sickle cell disease assciated target scores 

Goal: Identify which molecular targets have the strongest clinical evidence to HbF reactivation in sickle cell patients, beyond BCL11A.

In [3]:
import requests
import json
import pandas as pd

OT_API_URL = "https://api.platform.opentargets.org/api/v4/graphql"
test_query = """{meta {dataVersion {year month}}}"""
response = requests.post(OT_API_URL, json={"query": test_query})
print(response.status_code)
print(response.json())

200
{'data': {'meta': {'dataVersion': {'year': '25', 'month': '12'}}}}


In [18]:
diseaseId = "MONDO_0011382"
query = """ 
query SickleCell($diseaseId: String!, $page: Int!) {
  disease(efoId: $diseaseId) {
    name
    associatedTargets(page: {index: $page, size: 25}) {
      count
      rows {
        target {
          id
          approvedSymbol
          approvedName
        }
        score
        datatypeScores {
          score
        }
      }
    }
  }
}
"""

variables = {"diseaseId": diseaseId, "page": 0}
response = requests.post(OT_API_URL, json={"query": query, "variables": variables})
data = response.json()
print(f"Status: {response.status_code}")
print(f"Total targets found: {data['data']['disease']['associatedTargets']['count']}")
print(f"\nFirst 3 targets:")
for row in data['data']['disease']['associatedTargets']['rows'][:3]:
  print(f" {row['target']['approvedSymbol']} - {row['target']['approvedName']}- Score: {row['score']:.4f}")

Status: 200
Total targets found: 1207

First 3 targets:
 HBB - hemoglobin subunit beta- Score: 0.7961
 RRM2 - ribonucleotide reductase regulatory subunit M2- Score: 0.5756
 HBA1 - hemoglobin subunit alpha 1- Score: 0.5751


BCL11A not included in the genes with top scores let's me know that the above query pulled all genes associated with sickle cell disease. The expected target is appears lower in the ranking of the 1207 outputs. My research is focused on the reactivation of HbF in patients. Therefore, we are going to narrow down the query to answer that.

To do that, I will pull the full dataset rather than query only known targerts specifically because this is a discovery research, not confirmation.

In [13]:
all_targets = []
page = 0
while True:
    # update the page number in variables
    variables = {"diseaseId": diseaseId, "page": page}
    
    # make the API request (you already know how to do this)
    response = requests.post(OT_API_URL, json={"query": query, "variables": variables})
    data = response.json()
    
    # get the rows from the response
    rows = data['data']['disease']['associatedTargets']['rows']
    
    # if no rows come back, we've hit the last page
    if len(rows) == 0:
        break
    
    # otherwise add rows to our list and move to next page
    all_targets.extend(rows)
    page += 1
print(f"Total targets pulled: {len(all_targets)}")

Total targets pulled: 1207


In [19]:
df_targets = pd.DataFrame([{
    'target_id': row['target']['id'],
    'symbol': row['target']['approvedSymbol'],
    'name': row['target']['approvedName'],
    'score': row['score']
} for row in all_targets])

df_targets = df_targets.sort_values('score', ascending=False).reset_index(drop=True)

print(df_targets.shape)
print(df_targets.head(10))

(1207, 4)
         target_id  symbol                                               name  \
0  ENSG00000244734     HBB                            hemoglobin subunit beta   
1  ENSG00000171848    RRM2     ribonucleotide reductase regulatory subunit M2   
2  ENSG00000206172    HBA1                         hemoglobin subunit alpha 1   
3  ENSG00000188536    HBA2                         hemoglobin subunit alpha 2   
4  ENSG00000174175    SELP                                         selectin P   
5  ENSG00000169313  P2RY12                          purinergic receptor P2Y12   
6  ENSG00000112038   OPRM1                               opioid receptor mu 1   
7  ENSG00000169442    CD52                                      CD52 molecule   
8  ENSG00000176884   GRIN1  glutamate ionotropic receptor NMDA type subunit 1   
9  ENSG00000116032  GRIN3B  glutamate ionotropic receptor NMDA type subuni...   

      score  
0  0.796132  
1  0.575596  
2  0.575147  
3  0.575110  
4  0.556883  
5  0.352907  


From the look of the scores, there is a significant drop from HBB's score to RRM2's and the others. This is likely because this is the causal gene for the disorder ie the abnormal gene that causes SCD.

This highlights that in drug discovery, high asscoiation scores doesnt mean a good drug target. Association score ≠ druggability

In [17]:
df_targets.to_csv('../data/ot_scd_targets_raw.csv', index=False)

In [ ]:
# Find key HbF reactivation targets in the full dataset
hbf_targets = ['BCL11A', 'KLF1', 'MYC', 'ZBTB7A', 'MBD2', 'DNMT1']

df_hbf = df_targets[df_targets['symbol'].isin(hbf_targets)]
print(df_hbf[['symbol', 'name', 'score']])

      symbol                                      name     score
57     DNMT1                   DNA methyltransferase 1  0.085183
154     KLF1                KLF transcription factor 1  0.047079
461   BCL11A              BCL11 transcription factor A  0.016878
1027  ZBTB7A  zinc finger and BTB domain containing 7A  0.001848


From the above:
1. MYC and MBD@ do not have strong enough evidence in OpenTarget to apprear in the SCD association list
2. BCL11A, the target for the first approved CRISPR cure for the disorder, is ranked 461st with a score of 0.016878. This speaks to the idea that overall disease association is not sufficient on its own to identify Hbf reactivation targets.

The link is the disorder is recognised but it does not truly capture its usefulness to the mechanism we are looing at here.